# Cisco Foundation-Sec-8B Quantification & Inference Pipeline
This notebook demonstrates how to load and run the specialized **Cisco Foundation-Sec-8B-Reasoning** model on a standard cloud GPU environment (like Google Colab T4) using **4-bit quantization** to prevent system RAM crashes.

In [ ]:
# Step 1: Install required dependencies
!pip install -q transformers accelerate bitsandbytes huggingface_hub

In [ ]:
# Step 2: Import libraries and configure 4-bit Quantization
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Configure 4-bit quantization to fit the model within T4 GPU limits
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model_id = "fdtn-ai/Foundation-Sec-8B"

In [ ]:
# Step 3: Load Tokenizer and the Quantized Model
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model with 4-bit quantization (this avoids system RAM crash)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True
)
print("Model loaded successfully!")

In [ ]:
# Step 4: Run Inference with a CISA Threat Scenario
prompt = """<|begin_of_text|><|system|>
You are Metis, a cybersecurity reasoning model. Analyze the attack and map it to MITRE ATT&CK.
<|user|>
Palo Alto Networks PAN-OS contains an out-of-bounds write vulnerability in the User-ID Authentication Portal service that can allow an unauthenticated attacker to execute arbitrary code with root privileges by sending specially crafted packets.
<|assistant|>"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.2,
        top_p=0.95,
        do_sample=True
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)